# Agent Structured Output

The `structured_output.py` module defines the schema strategies, bindings, and exceptions used by LangChain agents to produce validated structured responses.

It supports Pydantic models, dataclasses, `TypedDict` classes, raw JSON Schema dictionaries, Python unions, and JSON Schema `oneOf` variants. Structured responses may be produced through model tool calls, provider-native JSON Schema support, or automatic strategy selection.

## Type Variables and Aliases

1. `SchemaT`: Represents the Python type produced after structured-output validation.
   * **Definition:**
     ```python
     SchemaT = TypeVar(
         "SchemaT"
     )
     ```

2. `SchemaKind`: Identifies the supported schema representation used for parsing.
   * **Definition:**
     ```python
     SchemaKind = Literal[
         "pydantic",
         "dataclass",
         "typeddict",
         "json_schema"
     ]
     ```

3. `ResponseFormat`: Represents every supported structured-output strategy.
   * **Definition:**
     ```python
     ResponseFormat = (
         ToolStrategy[
             SchemaT
         ]
         | ProviderStrategy[
             SchemaT
         ]
         | AutoStrategy[
             SchemaT
         ]
     )
     ```

# StructuredOutputError

`StructuredOutputError` is the base exception for errors produced while processing an agent's structured response.

## Bases

- `Exception`

## Attributes

1. `ai_message`: Stores the model message associated with the structured-output failure.
   * **Type:**
     ```python
     ai_message: AIMessage
     ```

# MultipleStructuredOutputsError

`MultipleStructuredOutputsError` is raised when a model returns more than one structured-output tool call even though the agent expects exactly one structured response.

## Bases

- `StructuredOutputError`

## Attributes

1. `tool_names`: Stores the names of the structured-output tools called by the model.
   * **Type:**
     ```python
     tool_names: list[str]
     ```

2. `ai_message`: Stores the model message containing the conflicting tool calls.
   * **Type:**
     ```python
     ai_message: AIMessage
     ```

### Methods

1. `__init__`: Creates an error describing the unexpected structured-output tool calls.

   The generated exception message lists every tool name and states that only one structured response was expected.

   * **Syntax:**
     ```python
     __init__(
         self,
         tool_names: list[str], # Names of the structured-output tools called
         ai_message: AIMessage # Message containing the conflicting tool calls
     ) -> None
     ```

# StructuredOutputValidationError

`StructuredOutputValidationError` is raised when arguments produced for a structured-output tool cannot be parsed according to its schema.

## Bases

- `StructuredOutputError`

## Attributes

1. `tool_name`: Stores the name of the structured-output tool whose arguments failed validation.
   * **Type:**
     ```python
     tool_name: str
     ```

2. `source`: Stores the original parsing or validation exception.
   * **Type:**
     ```python
     source: Exception
     ```

3. `ai_message`: Stores the model message containing the invalid structured output.
   * **Type:**
     ```python
     ai_message: AIMessage
     ```

### Methods

1. `__init__`: Creates a validation error while preserving the original exception and model message.

   The generated exception message identifies the failing tool and includes the source exception text.

   * **Syntax:**
     ```python
     __init__(
         self,
         tool_name: str, # Name of the structured-output tool
         source: Exception, # Original parsing or validation error
         ai_message: AIMessage # Message containing the invalid output
     ) -> None
     ```

# ToolStrategy

`ToolStrategy` configures structured output through artificial tool calls.

One or more output schemas are converted into schema specifications. A union type or JSON Schema containing `oneOf` is recursively expanded into separate leaf schemas, allowing the model to choose among multiple structured-response variants.

## Bases

- `Generic[SchemaT]`

## Attributes

1. `schema`: Stores the original schema supplied for tool-based structured output.

   The value may be a Pydantic model class, dataclass type, `TypedDict` class, raw JSON Schema dictionary, or Python union type.

   * **Type:**
     ```python
     schema: type[
         SchemaT
     ] | UnionType | dict[
         str,
         Any
     ]
     ```

2. `schema_specs`: Stores one normalized schema specification for each leaf response variant.

   Python unions and nested JSON Schema `oneOf` definitions are flattened recursively.

   * **Type:**
     ```python
     schema_specs: list[
         _SchemaSpec[
             Any
         ]
     ]
     ```

3. `tool_message_content`: Stores optional content placed in the `ToolMessage` returned after a valid artificial structured-output tool call.

   When `None`, the agent factory generates its default confirmation content.

   * **Type:**
     ```python
     tool_message_content: str | None
     ```

4. `handle_errors`: Controls which structured-output errors should be converted into retry messages.

   Supported values behave as follows:

   - `True` catches every structured-output error and uses the default message.
   - A string catches every error and uses that string.
   - An exception class catches only matching exceptions and uses the default message.
   - A tuple of exception classes catches matching exceptions and uses the default message.
   - A callable receives the exception and returns the retry message.
   - `False` disables structured-output retries and allows exceptions to propagate.

   * **Type:**
     ```python
     handle_errors: (
         bool
         | str
         | type[Exception]
         | tuple[
             type[Exception],
             ...
         ]
         | Callable[
             [Exception],
             str
         ]
     )
     ```

### Methods

1. `__init__`: Creates a tool-based structured-output strategy.

   The supplied schema is recursively expanded when it is a Python union or contains JSON Schema `oneOf` variants. Every resulting leaf schema is normalized into a schema specification.

   Unsupported leaf schema types cause schema normalization to raise `ValueError`.

   * **Syntax:**
     ```python
     __init__(
         self,
         schema: type[
             SchemaT
         ] | UnionType | dict[
             str,
             Any
         ], # Structured-output schema or union
         *,
         tool_message_content: str | None = None, # Tool-message content after successful parsing
         handle_errors: (
             bool
             | str
             | type[Exception]
             | tuple[
                 type[Exception],
                 ...
             ]
             | Callable[
                 [Exception],
                 str
             ]
         ) = True # Structured-output retry policy
     ) -> None
     ```

# ProviderStrategy

`ProviderStrategy` configures provider-native structured output.

The strategy converts a supported schema into the JSON Schema response-format payload expected by providers using the OpenAI-compatible structured-output shape.

## Bases

- `Generic[SchemaT]`

## Attributes

1. `schema`: Stores the original schema supplied for provider-native structured output.
   * **Type:**
     ```python
     schema: type[
         SchemaT
     ] | dict[
         str,
         Any
     ]
     ```

2. `schema_spec`: Stores the normalized schema name, description, kind, JSON Schema representation, and strictness setting.
   * **Type:**
     ```python
     schema_spec: _SchemaSpec[
         SchemaT
     ]
     ```

### Methods

1. `__init__`: Creates a provider-native structured-output strategy.

   Supported schemas are Pydantic model classes, dataclass types, `TypedDict` classes, and raw JSON Schema dictionaries. Unsupported schema types raise `ValueError`.

   * **Syntax:**
     ```python
     __init__(
         self,
         schema: type[
             SchemaT
         ] | dict[
             str,
             Any
         ], # Schema enforced by the model provider
         *,
         strict: bool | None = None # Request strict provider-side validation
     ) -> None
     ```

2. `to_model_kwargs`: Converts the strategy into model-binding keyword arguments.

   The returned payload uses `"type": "json_schema"` and includes the normalized schema name and JSON Schema. `"strict": True` is added only when strict mode is truthy.

   * **Syntax:**
     ```python
     to_model_kwargs(
         self
     ) -> dict[
         str,
         Any
     ]
     ```

# OutputToolBinding

`OutputToolBinding` associates one structured-output schema with the artificial LangChain tool used to expose it to a model.

It also retains the schema kind so tool-call arguments can be converted into the correct Python response type.

## Bases

- `Generic[SchemaT]`

## Attributes

1. `schema`: Stores the original Pydantic, dataclass, `TypedDict`, or JSON Schema definition.
   * **Type:**
     ```python
     schema: type[
         SchemaT
     ] | dict[
         str,
         Any
     ]
     ```

2. `schema_kind`: Stores how the schema should be validated and reconstructed.
   * **Type:**
     ```python
     schema_kind: SchemaKind
     ```

3. `tool`: Stores the `StructuredTool` exposed to the model for this response schema.
   * **Type:**
     ```python
     tool: BaseTool
     ```

### Methods

1. `from_schema_spec`: Creates a binding from a normalized schema specification.

   The generated `StructuredTool` receives the schema's JSON Schema as `args_schema`, along with its normalized name and description.

   * **Syntax:**
     ```python
     @classmethod
     from_schema_spec(
         cls,
         schema_spec: _SchemaSpec[
             SchemaT
         ] # Normalized schema specification
     ) -> Self
     ```

2. `parse`: Validates tool-call arguments according to the original schema.

   Pydantic models, dataclasses, and `TypedDict` classes are parsed through Pydantic's `TypeAdapter`. A raw JSON Schema dictionary returns the supplied argument dictionary unchanged because it has no Python type to instantiate.

   Validation failures are wrapped in `ValueError` with the schema name and original error text.

   * **Syntax:**
     ```python
     parse(
         self,
         tool_args: dict[
             str,
             Any
         ] # Arguments produced by the structured-output tool call
     ) -> SchemaT | dict[
         str,
         Any
     ]
     ```

# ProviderStrategyBinding

`ProviderStrategyBinding` associates a provider-native response schema with the parsing logic used after the model returns an `AIMessage`.

The binding extracts textual content, parses it as JSON, and validates the resulting dictionary according to the original schema.

## Bases

- `Generic[SchemaT]`

## Attributes

1. `schema`: Stores the original Pydantic, dataclass, `TypedDict`, or JSON Schema definition.
   * **Type:**
     ```python
     schema: type[
         SchemaT
     ] | dict[
         str,
         Any
     ]
     ```

2. `schema_kind`: Stores how the parsed JSON should be validated and reconstructed.
   * **Type:**
     ```python
     schema_kind: SchemaKind
     ```

### Methods

1. `from_schema_spec`: Creates a provider binding from a normalized schema specification.

   Only the original schema and its classified schema kind are retained.

   * **Syntax:**
     ```python
     @classmethod
     from_schema_spec(
         cls,
         schema_spec: _SchemaSpec[
             SchemaT
         ] # Normalized schema specification
     ) -> Self
     ```

2. `parse`: Extracts, decodes, and validates provider-native structured output.

   String message content is used directly. List content is concatenated by:

   - Reading `"text"` from dictionaries whose `"type"` is `"text"`.
   - Reading string `"content"` fields from other dictionaries.
   - Converting non-dictionary values to strings.
   - Ignoring dictionaries that contain neither supported text field.

   The concatenated text is decoded with `json.loads`. Invalid JSON raises `ValueError` identifying the expected schema. Valid JSON is then parsed according to `schema_kind`.

   Raw JSON Schema bindings return the decoded dictionary unchanged. Other schema kinds are validated through `TypeAdapter`, with validation failures wrapped in `ValueError`.

   * **Syntax:**
     ```python
     parse(
         self,
         response: AIMessage # Model message containing provider-native JSON
     ) -> SchemaT | dict[
         str,
         Any
     ]
     ```

# AutoStrategy

`AutoStrategy` stores a schema while allowing the agent factory to select the structured-output mechanism at runtime.

The factory may choose provider-native structured output when the selected model supports it or fall back to artificial tool calling otherwise.

## Bases

- `Generic[SchemaT]`

## Attributes

1. `schema`: Stores the schema used by the automatically selected response strategy.
   * **Type:**
     ```python
     schema: type[
         SchemaT
     ] | dict[
         str,
         Any
     ]
     ```

### Methods

1. `__init__`: Creates an automatic structured-output strategy.
   * **Syntax:**
     ```python
     __init__(
         self,
         schema: type[
             SchemaT
         ] | dict[
             str,
             Any
         ] # Schema used for automatic strategy selection
     ) -> None
     ```

## Supported Schema Forms

The module recognizes the following schema forms:

1. Pydantic model classes are classified as `"pydantic"` and converted through `model_json_schema`.

2. Dataclass types are classified as `"dataclass"` and converted through `TypeAdapter.json_schema`.

3. `TypedDict` classes are classified as `"typeddict"` and converted through `TypeAdapter.json_schema`.

4. Raw dictionaries are classified as `"json_schema"` and used directly as their JSON Schema representation.

Schema names are selected from an explicit name, a JSON Schema `"title"`, or the Python type's `__name__`. When no usable name exists, a generated `response_format_<suffix>` name is used.

Schema descriptions are selected from an explicit description, a JSON Schema `"description"`, or the Python type's docstring.

## Internal Components Omitted

The following private implementation details are not documented as standalone public API entries:

- `_SchemaSpec`
- `_parse_with_schema`
- `ProviderStrategyBinding._extract_text_content_from_message`
- Nested union and `oneOf` iteration helper